# RA3 — Análise, Visualização e Aprendizado de Máquina

## 1. Introdução

Este projeto investiga fatores demográficos, clínicos e relacionados à hospitalização que podem estar associados às readmissões hospitalares em até 30 dias após a alta de pacientes com diabetes.

Utilizando a base de dados preparada durante o RA1, esta etapa contempla análises estatísticas descritivas, visualizações e a implementação de um modelo de aprendizado de máquina supervisionado.

O objetivo é identificar padrões nos atendimentos hospitalares históricos e avaliar se as características dos pacientes e das internações podem contribuir para a previsão de readmissões precoces.

### Pergunta central

Quais fatores demográficos, clínicos e relacionados à hospitalização estão associados às readmissões hospitalares em até 30 dias após a alta de pacientes com diabetes?

### Objetivos

- Descrever as principais características e distribuições da base de dados.
- Investigar associações entre características dos pacientes, fatores de hospitalização, informações sobre medicamentos e readmissões precoces.
- Desenvolver visualizações para identificar padrões e diferenças relevantes.
- Aplicar um algoritmo de classificação supervisionada para prever readmissões em até 30 dias.
- Avaliar o desempenho do modelo e discutir as limitações dos resultados.

## 2. Perguntas analíticas

A análise será orientada pelas seguintes perguntas específicas:

1. **Tempo de internação:** Como a duração da internação (`time_in_hospital`) se relaciona com as taxas de readmissão em até 30 dias?

2. **Utilização anterior dos serviços de saúde:** Em que medida as internações, emergências e consultas ambulatoriais anteriores (`number_inpatient`, `number_emergency` e `number_outpatient`) indicam maior probabilidade de readmissão precoce?

3. **Alterações na medicação:** As alterações nos tratamentos e as informações sobre medicamentos para diabetes (`change`, `diabetesMed` e `insulin`) estão associadas às readmissões precoces?

4. **Diferenças demográficas e diagnósticas:** Existem diferenças nos padrões de readmissão precoce entre as faixas etárias (`age`) e as categorias dos diagnósticos principais (`diag_1`)?

In [16]:
# Importação das bibliotecas e carregamento dos dados

from pathlib import Path

import pandas as pd
import numpy as np

# Localizar o arquivo preparado no RA1
caminhos = [
    Path("data/processed/diabetic_data_clean.csv"),
    Path("../data/processed/diabetic_data_clean.csv")
]

caminho_dados = next(
    (caminho for caminho in caminhos if caminho.exists()),
    None
)

# Carregar a base preparada
df = pd.read_csv(caminho_dados)

print("Base carregada com sucesso!")
print("Dimensões da base:", df.shape)

Base carregada com sucesso!
Dimensões da base: (101766, 48)


In [17]:
# Verificação de valores ausentes

valores_ausentes = df.isnull().sum()

resumo_ausentes = pd.DataFrame({
    "Valores ausentes": valores_ausentes,
    "Percentual (%)": (valores_ausentes / len(df)) * 100
})

resumo_ausentes = resumo_ausentes[
    resumo_ausentes["Valores ausentes"] > 0
].sort_values(by="Percentual (%)", ascending=False)

display(resumo_ausentes)

,Valores ausentes,Percentual (%)
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
race,2273,2.233555
diag_3,1423,1.398306
diag_2,358,0.351787
diag_1,21,0.020636


In [18]:
# Distribuição da variável-alvo

distribuicao_target = df["readmitido_30_dias"].value_counts().sort_index()

resumo_target = pd.DataFrame({
    "Classificação": [
        "Sem readmissão em até 30 dias",
        "Readmissão em até 30 dias"
    ],
    "Quantidade": [
        distribuicao_target.get(0, 0),
        distribuicao_target.get(1, 0)
    ]
})

resumo_target["Percentual (%)"] = (
    resumo_target["Quantidade"] / len(df) * 100
)

display(resumo_target)

,Classificação,Quantidade,Percentual (%)
0,Sem readmissão em até 30 dias,90409,88.840084
1,Readmissão em até 30 dias,11357,11.159916


### Considerações iniciais sobre a base de dados

A base utilizada nesta etapa corresponde ao conjunto de dados preparado durante o RA1, contendo 101.766 registros de atendimentos hospitalares e 48 variáveis.

A verificação dos valores ausentes identificou maior concentração nas variáveis `max_glu_serum` e `A1Cresult`, enquanto as demais variáveis apresentam proporções menores de dados ausentes.

A distribuição da variável-alvo demonstra um desequilíbrio entre as classes: 88,84% dos atendimentos não apresentaram readmissão em até 30 dias, enquanto 11,16% apresentaram readmissão nesse período.

Esse desequilíbrio será considerado na construção e avaliação do modelo de classificação, uma vez que a acurácia isoladamente pode não representar adequadamente sua capacidade de identificar os casos de readmissão precoce.

## 3. Análise estatística descritiva

Nesta seção, serão utilizadas medidas estatísticas descritivas para compreender as características e a distribuição dos dados relacionados às internações hospitalares e ao histórico de utilização dos serviços de saúde.

Serão analisadas medidas de tendência central, como média e mediana, e medidas de dispersão, como desvio-padrão, valores mínimos e máximos e quartis.

A análise também considerará a frequência das categorias das variáveis qualitativas, permitindo identificar características predominantes na população estudada.

As variáveis foram selecionadas de acordo com sua relação com as perguntas analíticas definidas para o projeto.

In [20]:
# Estatísticas descritivas das variáveis numéricas

variaveis_numericas = [
    "time_in_hospital",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
    "number_diagnoses"
]


#metodo describe calcula automaticamente as estatísticas descritivas das variáveis numéricas
estatisticas_numericas = df[variaveis_numericas].describe().T

estatisticas_numericas = estatisticas_numericas.rename(columns={
    "count": "Quantidade",
    "mean": "Média",
    "std": "Desvio padrão",
    "min": "Mínimo",
    "25%": "1º quartil", #valor abaixo do qual 25% dos dados estão localizados
    "50%": "Mediana",
    "75%": "3º quartil", #valor abaixo do qual 75% dos dados estão localizados
    "max": "Máximo"
})

display(estatisticas_numericas.round(2))

,Quantidade,Média,Desvio padrão,Mínimo,1º quartil,Mediana,3º quartil,Máximo
time_in_hospital,101766.0,4.40,2.99,1.0,2.0,4.0,6.0,14.0
number_inpatient,101766.0,0.64,1.26,0.0,0.0,0.0,1.0,21.0
number_emergency,101766.0,0.20,0.93,0.0,0.0,0.0,0.0,76.0
number_outpatient,101766.0,0.37,1.27,0.0,0.0,0.0,0.0,42.0
number_diagnoses,101766.0,7.42,1.93,1.0,6.0,8.0,9.0,16.0


In [ ]:
# Comparação das variáveis numéricas por ocorrência de readmissão
#std = desvio padrão

comparacao_readmissao = (
    df.groupby("readmitido_30_dias")[variaveis_numericas]
    .agg(["mean", "median", "std"])
)

comparacao_readmissao.index = [
    "Sem readmissão em até 30 dias",
    "Readmissão em até 30 dias"
]

display(comparacao_readmissao.round(2))

time_in_hospital              number_inpatient  \
                                          mean median   std             mean   
Sem readmissão em até 30 dias             4.35    4.0  2.98             0.56   
Readmissão em até 30 dias                 4.77    4.0  3.03             1.22   

                                           number_emergency               \
                              median   std             mean median   std   
Sem readmissão em até 30 dias    0.0  1.13             0.18    0.0  0.86   
Readmissão em até 30 dias        0.0  1.95             0.36    0.0  1.37   

                              number_outpatient              number_diagnoses  \
                                           mean median   std             mean   
Sem readmissão em até 30 dias              0.36    0.0  1.26             7.39   
Readmissão em até 30 dias                  0.44    0.0  1.30             7.69   

                                            
                              median   std  
Sem readmissão em até 30 dias    8.0  1.95  
Readmissão em até 30 dias        9.0  1.77

### Análise das variáveis categóricas

Além das variáveis numéricas, serão analisadas características categóricas dos pacientes e dos tratamentos, incluindo faixa etária, gênero, alterações na medicação e utilização de insulina.

Para essas variáveis, serão utilizadas tabelas de frequência absoluta e relativa, permitindo identificar as categorias mais recorrentes na base de dados.

In [10]:
# 12. Distribuição dos atendimentos por faixa etária

distribuicao_idade = (
    df["age"]
    .value_counts()
    .reindex([
        "[0-10)", "[10-20)", "[20-30)", "[30-40)",
        "[40-50)", "[50-60)", "[60-70)", "[70-80)",
        "[80-90)", "[90-100)"
    ])
    .dropna()
)

resumo_idade = pd.DataFrame({
    "Quantidade": distribuicao_idade,
    "Percentual (%)": distribuicao_idade / len(df) * 100
})

display(resumo_idade.round(2))

,Quantidade,Percentual (%)
age,,
[0-10),161,0.16
[10-20),691,0.68
[20-30),1657,1.63
[30-40),3775,3.71
[40-50),9685,9.52
[50-60),17256,16.96
[60-70),22483,22.09
[70-80),26068,25.62
[80-90),17197,16.90


In [ ]:
# 13. Distribuição das variáveis relacionadas à medicação

#No = não houve alteração na medicação
#Ch = houve alteração na medicação

variaveis_medicacao = ["change", "diabetesMed", "insulin"]

for variavel in variaveis_medicacao:
    print(f"\nDistribuição da variável: {variavel}")

    frequencia = df[variavel].value_counts(dropna=False)

    resumo = pd.DataFrame({
        "Quantidade": frequencia,
        "Percentual (%)": frequencia / len(df) * 100
    })

    display(resumo.round(2))


Distribuição da variável: change


,Quantidade,Percentual (%)
change,,
No,54755,53.8
Ch,47011,46.2



Distribuição da variável: diabetesMed


,Quantidade,Percentual (%)
diabetesMed,,
Yes,78363,77.0
No,23403,23.0



Distribuição da variável: insulin


,Quantidade,Percentual (%)
insulin,,
No,47383,46.56
Steady,30849,30.31
Down,12218,12.01
Up,11316,11.12
